In [1]:
from ultralytics import YOLO
import cv2
import os
import numpy as np
import math
from scipy.signal import butter, filtfilt

from matplotlib import pyplot as plt

First we want to load our model. The path below is where mine was, fill in with your own

In [2]:
model = YOLO("./runs/detect/train7/weights/best.pt")

Now we read all the filenames from the data we're applying the model to. Change path here accordingly.

In [ ]:
# Read images from folder
img_path = "./measurements/m3/"
files = os.listdir(img_path + "frames/")
files = [f for f in files if "frame" in f]
files.sort()

print(len(files))

3082


Now for each file we apply the model to it, get the id and centre of each box found, and save them in an array for later use. We also draw the boxes and ids onto the images they came from and save those in the annotated folder.

In [ ]:
centres = []
for i in range(len(files)): 
    centres_i = []
    frame = model.track(img_path + "frames/" + files[i], verbose=False, conf=0.5, persist=True)
    annotated_frame = frame[0].plot()
    cv2.imwrite(img_path + "annotated/" + f"frame_{i:05d}.png", annotated_frame)
    for result in frame:
        boxes = result.boxes
        try:
            track_ids = result.boxes.id.int().cpu().tolist()
            for box, track_id in zip(boxes, track_ids):
                x1, y1, x2, y2 = box.xyxy[0].tolist()
                centres_i.append([track_id, (x1+x2)/2, (y1+y2)/2])
            
        except:
            print(files[i])
            
    centres.append(centres_i)

frame_00697.png
frame_00698.png


Here we fill any empty rows in the centres array with zeros instead of []. This makes the array parse nicely.

In [42]:
centres = [[[0, 0, 0]] if x==[] else x for x in centres]
for i in range(len(centres)):
    for j in range(len(centres[i])):
        if centres[i][j] == [0, 0, 0]:
            print("[" + str(i) + "][" + str(j) + "]: " + str(centres[i][j]))
print(centres[0])
len(centres)

[696][0]: [0, 0, 0]
[697][0]: [0, 0, 0]
[[1, 543.8843688964844, 173.42351531982422], [2, 551.4558715820312, 403.5216979980469], [3, 1057.1171569824219, 320.2179412841797], [22, 166.39062118530273, 379.4040222167969]]


3082

Here we sort everything in centres into time series, sorted by id. At the end of this pairs should contain as many columns as there are ids, with each column containing all (x,y) pairs relating to that id. When a new id shows up part way into the measurement, the column is back-filled with the first (x,y) pair relating to it, so that until it exists it does not appear to show movement, which a jump from (0,0) would.

In [ ]:
pairs = []
last_line = centres[0].copy()
ids_found = {}

for pair in last_line:
    if pair[0] != 0:
        pairs.append([pair[1:].copy()])
        ids_found[pair[0]] = len(pairs) - 1

for i in range(1, len(centres)):
    written_to = []
    for pair in centres[i]:
        if pair[0] in ids_found.keys():
            pairs[ids_found[pair[0]]].append(pair[1:].copy())
            written_to.append(ids_found[pair[0]])
        elif pair[0] != 0:
            if 0 in written_to:
                new_line = [pair[1:]] * (len(pairs[0]))
            else:
                new_line = [pair[1:]] * (len(pairs[0]) + 1)
            pairs.append(new_line)
            written_to.append(len(pairs) - 1)
            ids_found[pair[0]] = len(pairs) - 1
    for j in range(len(pairs)):
        if j not in written_to:
            pairs[j].append(pairs[j][-1])

Based on the location of the (x,y) pairs, each id is assigned to a shelf.

In [44]:
# assign to shelf 1, 2 or 3

shelf_height = 720/3
shelf_1 = []
shelf_2 = []
shelf_3 = []
for i in range(len(pairs)):
    if pairs[i][0][1] <= shelf_height:
        shelf_1.append(pairs[i])
        print(str(i) + ": shelf 1")
        print(pairs[i][0])
    elif pairs[i][0][1] > shelf_height and pairs[i][0][1] <= shelf_height*2:
        shelf_2.append(pairs[i])
        print(str(i) + ": shelf 2")
        print(pairs[i][0])
    else:
        shelf_3.append(pairs[i])
        print(str(i) + ": shelf 3")

0: shelf 1
[543.8843688964844, 173.42351531982422]
1: shelf 2
[551.4558715820312, 403.5216979980469]
2: shelf 2
[1057.1171569824219, 320.2179412841797]
3: shelf 2
[166.39062118530273, 379.4040222167969]
4: shelf 1
[1041.0132751464844, 112.84539794921875]
5: shelf 1
[1028.8975830078125, 105.8804702758789]
6: shelf 1
[1072.3520202636719, 168.10956573486328]
7: shelf 2
[176.2652359008789, 361.3403015136719]
8: shelf 2
[126.64197158813477, 390.8585205078125]
9: shelf 1
[1047.5118103027344, 197.43042755126953]
10: shelf 1
[1028.0628356933594, 171.197509765625]
11: shelf 1
[1031.0670776367188, 174.81553649902344]
12: shelf 1
[1046.504150390625, 194.92499542236328]


np arrays are created of the size to hold the mean vector norms of each shelf.

In [45]:
# calculate vector norm

if len(shelf_1) > 0:
    vectors_1 = np.empty((len(shelf_1), len(shelf_1[0])-1))
else:
    vectors_1 = None
if len(shelf_2) > 0:
    vectors_2 = np.empty((len(shelf_2), len(shelf_2[0])-1))
else:
    vectors_2 = None
if len(shelf_3) > 0:
    vectors_3 = np.empty((len(shelf_3), len(shelf_3[0])-1))
else:
    vectors_3 = None

Then each stream has the vector norms calculated.

In [46]:
if vectors_1 is not None:
    for i in range(len(shelf_1)):
        for j in range(len(shelf_1[0])-1):
            vectors_1[i, j] = math.sqrt(math.pow((shelf_1[i][j+1][0] - shelf_1[i][j][0]), 2) + math.pow((shelf_1[i][j+1][1] - shelf_1[i][j][1]), 2))
if vectors_2 is not None:
    for i in range(len(shelf_2)):
        for j in range(len(shelf_2[0])-1):
            vectors_2[i, j] = math.sqrt(math.pow((shelf_2[i][j+1][0] - shelf_2[i][j][0]), 2) + math.pow((shelf_2[i][j+1][1] - shelf_2[i][j][1]), 2))
if vectors_3 is not None:
    for i in range(len(shelf_3)):
        for j in range(len(shelf_3[0])-1):
            vectors_3[i, j] = math.sqrt(math.pow((shelf_3[i][j+1][0] - shelf_3[i][j][0]), 2) + math.pow((shelf_3[i][j+1][1] - shelf_3[i][j][1]), 2))

Here the mean vector norm is calculated, with a window size of 30s. Then the results are saved to csv files for loading into the local python script and plotting.

In [48]:
# calculate sliding vector norm
step_size = 10 # 1 second for a 10 Hz frame rate
window = 30 # thirty second window size

mean_vectors_1 = []
if vectors_1 is not None:
    for i in range(vectors_1.shape[0]):
        mean_vectors_1.append(np.divide(np.convolve(vectors_1[i, :], np.ones(window*step_size), 'same'), window*step_size))
        # mean_vectors[i] = (mean_vectors[i] - mean_vectors[i].min()) / (mean_vectors[i].max() - mean_vectors[i].min())
mean_vectors_2 = []
if vectors_2 is not None:
    for i in range(vectors_2.shape[0]):
        mean_vectors_2.append(np.divide(np.convolve(vectors_2[i, :], np.ones(window*step_size), 'same'), window*step_size))
        # mean_vectors[i] = (mean_vectors[i] - mean_vectors[i].min()) / (mean_vectors[i].max() - mean_vectors[i].min())
mean_vectors_3 = []
if vectors_3 is not None:
    for i in range(vectors_3.shape[0]):
        mean_vectors_3.append(np.divide(np.convolve(vectors_3[i, :], np.ones(window*step_size), 'same'), window*step_size))
        # mean_vectors[i] = (mean_vectors[i] - mean_vectors[i].min()) / (mean_vectors[i].max() - mean_vectors[i].min())
    
mean_vectors_1 = np.array(mean_vectors_1)
mean_vectors_2 = np.array(mean_vectors_2)
mean_vectors_3 = np.array(mean_vectors_3)
np.savetxt("./s1_sh1_m.csv", mean_vectors_1, delimiter=",")
np.savetxt("./s1_sh2_m.csv", mean_vectors_2, delimiter=",")
np.savetxt("./s1_sh3_m.csv", mean_vectors_3, delimiter=",")